In [2]:
import pandas as pd
import sqlite3
from pathlib import Path
from datetime import datetime, timezone

In [8]:
# Folder containing your downloaded AGMARKNET CSV files
DATA_DIR = Path("./raw")

# SQLite database file
DB_PATH = Path("../database/mandi_prices.db")
DB_PATH.parent.mkdir(parents=True, exist_ok=True)

# AGMARKNET export columns after using row 0 as the header
EXPECTED_COLUMNS = [
    "state",
    "district",
    "market",
    "commodity_group",
    "commodity",
    "variety",
    "grade",
    "min_price",
    "max_price",
    "modal_price",
    "price_unit",
    "arrival_quantity",
    "arrival_unit",
    "arrival_date",
]

In [9]:
def clean_number(value):
    """Convert strings such as '1,921.00' into 1921.0."""
    if pd.isna(value):
        return None

    value = str(value).replace(",", "").strip()
    return pd.to_numeric(value, errors="coerce")

In [10]:
def read_agmarknet_csv(file_path: Path) -> pd.DataFrame:
    # The first CSV row contains actual field names.
    df = pd.read_csv(file_path, header=0)

    # Rename columns by position because AGMARKNET puts report title
    # in the Min Price column name.
    df.columns = EXPECTED_COLUMNS

    # Remove blank / malformed rows
    df = df.dropna(subset=["state", "market", "commodity", "arrival_date"])

    # Standardize text
    text_columns = [
        "state", "district", "market", "commodity_group",
        "commodity", "variety", "grade", "price_unit", "arrival_unit"
    ]
    for col in text_columns:
        df[col] = df[col].astype(str).str.strip()

    # Convert numeric fields
    for col in ["min_price", "max_price", "modal_price", "arrival_quantity"]:
        df[col] = df[col].apply(clean_number)

    # AGMARKNET date format: 08-07-2026
    df["arrival_date"] = pd.to_datetime(
        df["arrival_date"],
        format="%d-%m-%Y",
        errors="coerce"
    ).dt.strftime("%Y-%m-%d")

    # Remove invalid dates / numeric rows
    df = df.dropna(subset=["arrival_date", "modal_price"])

    return df

In [11]:
# Read and combine every CSV in data/raw/
all_rows = []
for csv_file in DATA_DIR.glob("*.csv"):
    try:
        cleaned = read_agmarknet_csv(csv_file)
        all_rows.append(cleaned)
        print(f"Loaded {csv_file.name}: {len(cleaned)} valid rows")
    except Exception as e:
        print(f"Skipped {csv_file.name}: {e}")

Loaded Brinjal.csv: 37 valid rows
Loaded Cauliflower.csv: 35 valid rows
Loaded Green_Chilli.csv: 35 valid rows
Loaded Onion.csv: 1 valid rows
Loaded Potato.csv: 36 valid rows
Loaded Tomato.csv: 363 valid rows


In [12]:
if not all_rows:
    raise RuntimeError("No valid CSV rows found. Check DATA_DIR.")

In [13]:
df = pd.concat(all_rows, ignore_index=True)

In [14]:
df.head(5)

,state,district,market,commodity_group,commodity,variety,grade,min_price,max_price,modal_price,price_unit,arrival_quantity,arrival_unit,arrival_date
0,Andhra Pradesh,Chittor,Palamaner APMC,Vegetables,Brinjal,Brinjal,FAQ,3000.0,5000.0,4000.0,Rs./Quintal,1.9,Metric Tonnes,2026-06-30
1,Andhra Pradesh,Chittor,Palamaner APMC,Vegetables,Brinjal,Brinjal,FAQ,3000.0,5000.0,4000.0,Rs./Quintal,2.1,Metric Tonnes,2026-06-29
2,Andhra Pradesh,Chittor,Palamaner APMC,Vegetables,Brinjal,Brinjal,FAQ,3000.0,5000.0,4000.0,Rs./Quintal,1.7,Metric Tonnes,2026-06-28
3,Andhra Pradesh,Chittor,Palamaner APMC,Vegetables,Brinjal,Brinjal,FAQ,2000.0,4000.0,3000.0,Rs./Quintal,2.3,Metric Tonnes,2026-06-27
4,Andhra Pradesh,Chittor,Palamaner APMC,Vegetables,Brinjal,Brinjal,FAQ,4000.0,5000.0,4500.0,Rs./Quintal,1.6,Metric Tonnes,2026-06-26


In [15]:
len(df)

507

In [16]:
# Prevent duplicate observations across CSV files
dedupe_columns = [
    "arrival_date", "state", "district", "market",
    "commodity", "variety", "grade"
]
df = df.drop_duplicates(subset=dedupe_columns, keep="last")

In [17]:
len(df)

507

In [18]:
# Create database and table
conn = sqlite3.connect(DB_PATH)

In [19]:
conn.execute("DROP TABLE IF EXISTS mandi_market_data")

conn.execute("""
CREATE TABLE mandi_market_data (
    state TEXT NOT NULL,
    district TEXT,
    market TEXT NOT NULL,
    commodity_group TEXT,
    commodity TEXT NOT NULL,
    variety TEXT,
    grade TEXT,
    min_price REAL,
    max_price REAL,
    modal_price REAL NOT NULL,
    price_unit TEXT,
    arrival_quantity REAL,
    arrival_unit TEXT,
    arrival_date TEXT NOT NULL,

    PRIMARY KEY (
        arrival_date,
        state,
        district,
        market,
        commodity,
        variety,
        grade
    )
)
""")

conn.commit()
conn.close()

print("Old table deleted. New metadata-free table created.")

Old table deleted. New metadata-free table created.


In [20]:
rows = df.to_dict(orient="records")

In [21]:
len(rows)

507

In [23]:
conn = sqlite3.connect(DB_PATH)

conn.executemany("""
INSERT INTO mandi_market_data (
    state, district, market, commodity_group, commodity,
    variety, grade, min_price, max_price, modal_price,
    price_unit, arrival_quantity, arrival_unit, arrival_date
)
VALUES (
    :state, :district, :market, :commodity_group, :commodity,
    :variety, :grade, :min_price, :max_price, :modal_price,
    :price_unit, :arrival_quantity, :arrival_unit, :arrival_date
)
ON CONFLICT(
    arrival_date, state, district, market,
    commodity, variety, grade
)
DO UPDATE SET
    commodity_group = excluded.commodity_group,
    min_price = excluded.min_price,
    max_price = excluded.max_price,
    modal_price = excluded.modal_price,
    price_unit = excluded.price_unit,
    arrival_quantity = excluded.arrival_quantity,
    arrival_unit = excluded.arrival_unit
""", rows)

conn.commit()

total = conn.execute(
    "SELECT COUNT(*) FROM mandi_market_data"
).fetchone()[0]

print(f"Inserted/updated successfully. Total rows: {total}")

conn.close()

Inserted/updated successfully. Total rows: 507


In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../database/mandi_prices.db")


u_query = """
SELECT *
FROM mandi_market_data 
"""
history = pd.read_sql_query(
    u_query,
    conn,
)

print(history)
conn.close()

              state   district            market commodity_group  \
0    Andhra Pradesh    Chittor    Palamaner APMC      Vegetables   
1    Andhra Pradesh    Chittor    Palamaner APMC      Vegetables   
2    Andhra Pradesh    Chittor    Palamaner APMC      Vegetables   
3    Andhra Pradesh    Chittor    Palamaner APMC      Vegetables   
4    Andhra Pradesh    Chittor    Palamaner APMC      Vegetables   
..              ...        ...               ...             ...   
516  Andhra Pradesh    Chittor    Palamaner APMC      Vegetables   
517  Andhra Pradesh    Chittor    Palamaner APMC      Vegetables   
518  Andhra Pradesh    Chittor    Palamaner APMC      Vegetables   
519  Andhra Pradesh  Annamayya     Punganur APMC      Vegetables   
520  Andhra Pradesh  Annamayya  Madanapalli APMC      Vegetables   

        commodity       variety grade  min_price  max_price  modal_price  \
0         Brinjal       Brinjal   FAQ     3000.0     5000.0       4000.0   
1         Brinjal       Brinjal

All data from Ag Market is now successfully inserted into the SQLite database. 